# Model Development

This notebook covers baseline model development for the Customer Churn Prediction Platform.

The preprocessing pipeline developed during Phase 6 is reused to ensure consistent feature engineering and preprocessing during model training and evaluation.

## 7.1 Baseline Model

A Logistic Regression model is established as the baseline classifier.

The baseline provides a simple, interpretable reference point against which more advanced models can be evaluated.

In [1]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Reusable Preprocessing Pipeline

The preprocessing workflow developed during Phase 6 is implemented as a reusable production module.

The same pipeline will be used for model training and later inference to maintain consistency between development and deployment.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT_71 = Path.cwd().parent

if str(PROJECT_ROOT_71) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_71))

from backend.src.ml.preprocessing_pipeline import build_preprocessing_pipeline

pipeline_71 = build_preprocessing_pipeline()

print("Reusable preprocessing pipeline loaded successfully.")
print("Pipeline steps:", list(pipeline_71.named_steps.keys()))

Reusable preprocessing pipeline loaded successfully.
Pipeline steps: ['data_preparation', 'feature_engineering', 'preprocessing']


## Load Training Dataset

The raw customer churn dataset is loaded for baseline model development.

The target variable `Churn` is separated from the input features, while the reusable preprocessing pipeline handles data preparation, feature engineering, encoding, and scaling.

In [3]:
DATA_PATH_71 = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df_71 = pd.read_csv(DATA_PATH_71)

X_71 = df_71.drop(columns=["customerID", "Churn"])
y_71 = df_71["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("=" * 60)
print("BASELINE DATASET")
print("=" * 60)

print("Input shape:", X_71.shape)
print("Target shape:", y_71.shape)
print("Missing target values:", y_71.isnull().sum())

BASELINE DATASET
Input shape: (7043, 19)
Target shape: (7043,)
Missing target values: 0


## Train-Test Split

An 80/20 stratified split is used for baseline model development.

The split is performed before fitting the preprocessing pipeline so that all data-dependent transformations are learned only from the training data.

In [4]:
from sklearn.model_selection import train_test_split

X_train_71, X_test_71, y_train_71, y_test_71 = train_test_split(
    X_71,
    y_71,
    test_size=0.20,
    random_state=42,
    stratify=y_71
)

print("Training data:", X_train_71.shape)
print("Testing data:", X_test_71.shape)

Training data: (5634, 19)
Testing data: (1409, 19)


## Logistic Regression Baseline

Logistic Regression is used as the baseline classifier because it is simple, interpretable, computationally efficient, and provides a strong reference point for comparing subsequent models.

In [5]:
baseline_model_71 = Pipeline(
    steps=[
        ("preprocessing", pipeline_71),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

baseline_model_71.fit(
    X_train_71,
    y_train_71
)

print("Baseline Logistic Regression trained successfully.")

Baseline Logistic Regression trained successfully.


In [6]:
y_pred_71 = baseline_model_71.predict(X_test_71)
y_prob_71 = baseline_model_71.predict_proba(X_test_71)[:, 1]

print("=" * 60)
print("BASELINE MODEL PERFORMANCE")
print("=" * 60)

print("Accuracy :", accuracy_score(y_test_71, y_pred_71))
print("Precision:", precision_score(y_test_71, y_pred_71))
print("Recall   :", recall_score(y_test_71, y_pred_71))
print("F1 Score :", f1_score(y_test_71, y_pred_71))
print("ROC-AUC  :", roc_auc_score(y_test_71, y_prob_71))

BASELINE MODEL PERFORMANCE
Accuracy : 0.7998580553584103
Precision: 0.6554054054054054
Recall   : 0.5187165775401069
F1 Score : 0.5791044776119403
ROC-AUC  : 0.8424242424242425
